In [15]:
import os
from pathlib import Path
import sys

import pandas as pd
import pycountry_convert as pc
import plotly.express as px


In [16]:
# load data
project_root = Path.cwd().parent  # assumes you're in /notebooks
sys.path.append(str(project_root))

from backend.classes import CORDIS_data#, Project_data
horizon_data = CORDIS_data(parent_dir=project_root, enrich=True)
df = horizon_data.organization_df

Enriching the projects dataset with temporal information.
Enriching the projects dataset with people and institutions information.
Enriching the projects dataset with financial information.
Enriching the projects dataset with thematic / scientific information.
Enriched project_df with scientific and thematic information: 15863 projects
Columns of the project dataframe after enrichment:
  - 35 columns
  - Columns: id, acronym, status, title, startDate, endDate, totalCost, ecMaxContribution, legalBasis, topics, ecSignatureDate, frameworkProgramme, masterCall, subCall, fundingScheme, nature, objective, contentUpdateDate, rcn, grantDoi, duration_days, duration_months, duration_years, projectID_x, n_institutions, projectID_y, institutions, projectID, coordinator_name, ecContribution_per_year, totalCost_per_year, field_class, field, subfield, niche


In [17]:
# add country names
country_names = pd.read_csv(os.path.join(project_root,'data/raw/cordisref-countries.csv'), on_bad_lines='warn', delimiter=';', keep_default_na=False)
country_names = country_names[country_names['language']=='en'].drop(columns='language')
country_names = country_names.rename(columns={'name':'countryName'})
df = pd.merge(df, country_names, left_on='country', right_on = 'euCode', how='inner')#.drop(columns='Unnamed: 0')
df['organizationCount'] = 1

# add continent names and codes (using pycountry-convert)
def convert_to_continent(isocode):
    try:
        return pc.country_alpha2_to_continent_code(isocode)
    except:
        #print(isocode, 'not a valid code, set manually')
        return
df['continent'] = df.isoCode.apply(convert_to_continent)
exceptions_dict = {'BQ': 'SA', 'VA': 'EU', 'XK': 'EU'}
df['continent'] = df['continent'].fillna(df['euCode'].map(exceptions_dict)) # manually add a few without ISO codes

continent_dict = dict(AF='Africa', AN='Antarctica', AS='Asia', EU='Europe', NA='North America', OC='Oceania', SA='South America')
df['continentName'] = df.continent.map(continent_dict)

In [18]:
# choose metric
metrics_list = ['ecContribution', 'netEcContribution', 'totalCost', 'organizationCount']
metric = metrics_list[1]

# filter for participation role
role = 'any' # any, coordinator, participant, thirdParty, associatedPartner

if role!='any':
    df = df[df.role==role]

# For each country, keep top N cities and group the rest
N = 20
dfs = []
for (continent, country), group in df.groupby(['continentName', 'countryName']):
    group_total = group[metric].sum()
    
    top_cities = group.groupby('city')[metric].sum().sort_values(ascending=False)[:N].index
    top = group[group.city.isin(top_cities)]
    
    # 'other' contribution
    other_metric = group_total - top[metric].sum()
    if other_metric > 0:
        other_row = pd.DataFrame([{
            'continentName': continent,
            'countryName': country,
            'city': 'Other cities',
            metric: other_metric
            #'color':'grey'
        }])
        top = pd.concat([top, other_row])
    dfs.append(top)

df_plot = pd.concat(dfs)

In [19]:
print('role:',role, '\nplotting', metric)
fig = px.treemap(df_plot, 
                 path=[px.Constant('All'), 'continentName', 'countryName', 'city'], 
                 values=metric,
                 maxdepth=3
                 )
fig.update_traces(root_color="lightgrey")
fig.update_layout(margin = dict(t=50, l=25, r=25, b=25))
fig.show()

role: any 
plotting netEcContribution
